In [ ]:
# =========================================
# 1. INSTALLS
# =========================================
!pip install -q transformers accelerate sentencepiece tqdm

# =========================================
# 2. IMPORTS
# =========================================
import json
import re
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# =========================================
# 3. DEVICE SETUP (CRITICAL FIX)
# =========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================================
# 4. MODEL LOAD (FLAN-T5)
# =========================================
model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

# =========================================
# 5. LOAD STORIES
# =========================================
def load_stories(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    stories = re.split(r"Story\s*\d+[:.]\s*", text)
    stories = [s.strip() for s in stories if len(s.strip()) > 20]

    return stories

stories = load_stories("/content/all_stories.txt.txt")
print("Total stories:", len(stories))

# =========================================
# 6. PROMPT (ESCAPED BRACES FIXED)
# =========================================
PROMPT = """
Convert the story into structured JSON scenes ONLY.

STRICT RULES:
- Output ONLY valid JSON (no explanation)
- 3 to 5 scenes
- Same theme across all scenes
- Do NOT add extra keys

Allowed labels:

emotion: sad, happy, confused, fearful, calm
setting: home, office, street, market, temple, nature, space
atmosphere: tense, peaceful, neutral, chaotic
state: overwhelmed, hopeful, uncertain, confident
theme: failure, growth, purpose, resilience, self_discovery

Output format:

{{
  "scenes": [
    {{
      "scene_id": 1,
      "emotion": "",
      "setting": "",
      "atmosphere": "",
      "state": "",
      "theme": "",
      "state_change": ""
    }}
  ]
}}

Story:
{story}
"""

# =========================================
# 7. JSON EXTRACTION (ROBUST)
# =========================================
def extract_json(text):
    if not text:
        return None

    text = text.replace("```json", "").replace("```", "")

    matches = re.findall(r"\{.*\}", text, re.DOTALL)

    for m in reversed(matches):
        try:
            return json.loads(m)
        except:
            continue

    return None

# =========================================
# 8. NORMALIZATION (LABEL FIX)
# =========================================
LABELS = {
    "emotion": {"sad","happy","confused","fearful","calm"},
    "setting": {"home","office","street","market","temple","nature","space"},
    "atmosphere": {"tense","peaceful","neutral","chaotic"},
    "state": {"overwhelmed","hopeful","uncertain","confident"},
    "theme": {"failure","growth","purpose","resilience","self_discovery"}
}

def normalize(field, val):
    if not val:
        return None
    val = str(val).lower()
    for l in LABELS[field]:
        if l in val:
            return l
    return None

def clean_scene(s):
    return {
        "scene_id": s.get("scene_id"),
        "emotion": normalize("emotion", s.get("emotion")),
        "setting": normalize("setting", s.get("setting")),
        "atmosphere": normalize("atmosphere", s.get("atmosphere")),
        "state": normalize("state", s.get("state")),
        "theme": normalize("theme", s.get("theme")),
        "state_change": s.get("state_change")
    }

# =========================================
# 9. GENERATION FUNCTION
# =========================================
def generate_story(story, max_new_tokens=1024):

    prompt = PROMPT.format(story=story)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=4
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# =========================================
# 10. PIPELINE WITH CHECKPOINTING
# =========================================
out_path = "/content/flan_decomposed_output.json"

try:
    with open(out_path, "r") as f:
        results = json.load(f)
except:
    results = []

start = len(results)

for i in tqdm(range(start, len(stories))):

    story = stories[i]

    raw = generate_story(story)

    parsed = extract_json(raw)

    if parsed and "scenes" in parsed:
        scenes = [clean_scene(s) for s in parsed["scenes"]]
    else:
        scenes = None

    results.append({
        "id": i + 1,
        "input": story,
        "raw_output": raw,
        "parsed": scenes
    })

    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)

print("DONE. Saved at:", out_path)

Using device: cuda


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Total stories: 40


100%|██████████| 40/40 [03:03<00:00,  4.59s/it]

DONE. Saved at: /content/flan_decomposed_output.json


In [ ]:
import json
import re
import numpy as np
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ===============================
# 1. SAFE JSON / TEXT PARSER
# ===============================

def safe_extract(text):
    """
    Extract structured fields even from broken model outputs
    """
    if text is None:
        return []

    if isinstance(text, list):
        return text

    text = str(text)

    # try direct JSON
    try:
        return json.loads(text)
    except:
        pass

    # fallback: extract pseudo JSON blocks
    blocks = re.findall(r"\{.*?\}", text, re.DOTALL)

    parsed = []
    for b in blocks:
        try:
            parsed.append(json.loads(b))
        except:
            continue

    return parsed


# ===============================
# 2. NORMALIZATION
# ===============================

FIELDS = ["emotion", "setting", "atmosphere", "state", "theme"]

def normalize_scene(scene):
    norm = {}
    for f in FIELDS:
        val = scene.get(f, "")
        if isinstance(val, list):
            val = " ".join(val)
        norm[f] = str(val).lower().strip()
    return norm


# ===============================
# 3. FLATTEN SCENES
# ===============================

def flatten(stories):
    all_scenes = []
    for s in stories:
        scenes = safe_extract(s.get("output", []))
        for sc in scenes:
            sc["_story_id"] = s.get("id")
            all_scenes.append(sc)
    return all_scenes


# ===============================
# 4. SEMANTIC ALIGNMENT
# ===============================

def match_scenes(gt_scenes, pred_scenes):
    gt_texts = [json.dumps(normalize_scene(s)) for s in gt_scenes]
    pr_texts = [json.dumps(normalize_scene(s)) for s in pred_scenes]

    if len(gt_texts) == 0 or len(pr_texts) == 0:
        return []

    vec = TfidfVectorizer().fit(gt_texts + pr_texts)
    gt_vec = vec.transform(gt_texts)
    pr_vec = vec.transform(pr_texts)

    sim = cosine_similarity(gt_vec, pr_vec)

    matches = []
    used = set()

    for i in range(len(gt_scenes)):
        best_j = np.argmax(sim[i])
        if best_j not in used:
            matches.append((i, best_j))
            used.add(best_j)

    return matches


# ===============================
# 5. F1 CALCULATION
# ===============================

def f1(gt, pr):
    gt_set = set(gt.split())
    pr_set = set(pr.split())

    if len(gt_set) == 0 and len(pr_set) == 0:
        return 1.0

    if len(gt_set) == 0 or len(pr_set) == 0:
        return 0.0

    tp = len(gt_set & pr_set)
    precision = tp / len(pr_set)
    recall = tp / len(gt_set)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


# ===============================
# 6. MAIN EVALUATOR
# ===============================

def evaluate(gt_file, pred_file):

    gt = json.load(open(gt_file))
    pred = json.load(open(pred_file))

    gt_scenes = flatten(gt)
    pred_scenes = flatten(pred)

    matches = match_scenes(gt_scenes, pred_scenes)

    scores = {f: [] for f in FIELDS}

    for gi, pj in matches:
        g = normalize_scene(gt_scenes[gi])
        p = normalize_scene(pred_scenes[pj])

        for f in FIELDS:
            scores[f].append(f1(g[f], p[f]))

    print("\n================ FINAL RESULTS ================\n")

    for f in FIELDS:
        val = np.mean(scores[f]) if scores[f] else 0.0
        print(f"{f}_f1: {val:.4f}")


# ===============================
# RUN
# ===============================

evaluate(
    "/content/Testing_Ground_Truth_Data.txt",
    "/content/flan_decomposed_output.json"
)


================ FINAL RESULTS ================

emotion_f1: 0.0000
setting_f1: 0.0000
atmosphere_f1: 0.0000
state_f1: 0.0000
theme_f1: 0.0000


In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# =========================
# DEVICE FIX (IMPORTANT)
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================
# MODEL LOAD (FLAN-T5)
# =========================
model_name = "google/flan-t5-large"   # or flan-t5-xl if available

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

model.eval()

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
              (wi_1): Linear(in_features=1024, out_features=2816, bias=False)
       

In [ ]:
# =========================
# SAFE FIELD ACCESSOR
# =========================

def get_field(item, key, default=None):
    if isinstance(item, dict):
        return item.get(key, default)
    return default


# =========================
# RUN FULL DATASET (FIXED)
# =========================

results = []

print("Total stories:", len(data))

for i, item in enumerate(tqdm(data)):

    # safe extraction
    story = get_field(item, "input") or get_field(item, "story") or get_field(item, "text")

    if story is None:
        continue

    raw_output = generate_story(story)
    parsed_output = extract_json(raw_output)

    results.append({
        "id": i,   # FIX: auto-index instead of missing dataset id
        "input": story,
        "raw_output": raw_output,
        "parsed": parsed_output
    })


# =========================
# SAVE OUTPUT
# =========================

output_file = "/content/flan_decomposed_output.json"

with open(output_file, "w") as f:
    json.dump(results, f, indent=2)

print("DONE:", output_file)

Total stories: 40


100%|██████████| 40/40 [02:27<00:00,  3.69s/it]

DONE: /content/flan_decomposed_output.json


In [ ]:
import json

def load_gt(path):
    """
    Handles:
    - JSON list file
    - JSONL
    - mixed formatting
    """
    with open(path, "r") as f:
        raw = f.read().strip()

    # Case 1: full JSON array
    if raw.startswith("["):
        return json.loads(raw)

    # Case 2: JSONL fallback
    data = []
    for line in raw.split("\n"):
        line = line.strip().rstrip(",")
        if not line:
            continue
        try:
            data.append(json.loads(line))
        except:
            continue

    return data

In [ ]:
import json
import re
from difflib import SequenceMatcher
from tqdm import tqdm

# =========================================================
# SAFE LOAD GT (FIXED)
# =========================================================

def load_gt(path):
    with open(path, "r") as f:
        raw = f.read().strip()

    # Case 1: full JSON array (YOUR CASE)
    if raw.startswith("["):
        try:
            return json.loads(raw)
        except Exception:
            pass

    # Case 2: fallback broken JSON lines
    data = []
    buffer = ""
    for line in raw.split("\n"):
        buffer += line.strip()
        if buffer.endswith("}"):
            try:
                data.append(json.loads(buffer))
            except:
                pass
            buffer = ""

    return data


def load_pred(path):
    with open(path, "r") as f:
        return json.load(f)


# =========================================================
# SAFE TEXT EXTRACTION FROM FLAN OUTPUT
# =========================================================

def extract_field(text, key):
    if not isinstance(text, str):
        return ""
    pattern = rf'"{key}"\s*:\s*"([^"]*)"'
    match = re.search(pattern, text)
    return match.group(1).strip().lower() if match else ""


def extract_pred_fields(raw):
    return {
        "emotion": extract_field(raw, "emotion"),
        "setting": extract_field(raw, "setting"),
        "atmosphere": extract_field(raw, "atmosphere"),
        "state": extract_field(raw, "state"),
        "theme": extract_field(raw, "theme"),
    }


# =========================================================
# FUZZY MATCH
# =========================================================

def sim(a, b):
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def f1(gt_list, pred_list, thresh=0.6):
    tp = fp = fn = 0
    used = [False] * len(gt_list)

    for p in pred_list:
        matched = False
        for i, g in enumerate(gt_list):
            if used[i]:
                continue
            if sim(g, p) >= thresh:
                tp += 1
                used[i] = True
                matched = True
                break
        if not matched:
            fp += 1

    fn = used.count(False)

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    return 2 * precision * recall / (precision + recall + 1e-9)


def safe_mean(x):
    return sum(x)/len(x) if len(x) else 0.0


# =========================================================
# LOAD DATA
# =========================================================

gt_data = load_gt("/content/Testing_Ground_Truth_Data.txt")
pred_data = load_pred("/content/flan_decomposed_output.json")

print("GT:", len(gt_data), "Pred:", len(pred_data))


# =========================================================
# METRIC STORAGE
# =========================================================

emotion_scores = []
setting_scores = []
atmosphere_scores = []
state_scores = []
theme_scores = []


# =========================================================
# MAIN LOOP
# =========================================================

for i in tqdm(range(min(len(gt_data), len(pred_data)))):

    gt_story = gt_data[i].get("output", [])
    pred_raw = pred_data[i].get("raw_output", "")

    # GT extraction
    gt_emotion = [s.get("emotion", "").lower() for s in gt_story]
    gt_setting = [s.get("setting", "").lower() for s in gt_story]
    gt_atmosphere = [s.get("atmosphere", "").lower() for s in gt_story]
    gt_state = [s.get("state", "").lower() for s in gt_story]

    # PRED extraction
    pred_fields = extract_pred_fields(pred_raw)

    pred_emotion = [pred_fields["emotion"]]
    pred_setting = [pred_fields["setting"]]
    pred_atmosphere = [pred_fields["atmosphere"]]
    pred_state = [pred_fields["state"]]

    emotion_scores.append(f1(gt_emotion, pred_emotion))
    setting_scores.append(f1(gt_setting, pred_setting))
    atmosphere_scores.append(f1(gt_atmosphere, pred_atmosphere))
    state_scores.append(f1(gt_state, pred_state))

    # theme (story-level)
    gt_theme = max(set([s.get("theme","").lower() for s in gt_story]), default="")
    pred_theme = pred_fields["theme"]

    theme_scores.append(sim(gt_theme, pred_theme))


# =========================================================
# FINAL RESULTS
# =========================================================

print("\n================ FINAL RESULTS ================\n")

print("emotion_f1:", round(safe_mean(emotion_scores), 4))
print("setting_f1:", round(safe_mean(setting_scores), 4))
print("atmosphere_f1:", round(safe_mean(atmosphere_scores), 4))
print("state_f1:", round(safe_mean(state_scores), 4))
print("theme_f1:", round(safe_mean(theme_scores), 4))

GT: 40 Pred: 40


100%|██████████| 40/40 [00:00<00:00, 7765.43it/s]


================ FINAL RESULTS ================

emotion_f1: 0.0125
setting_f1: 0.0125
atmosphere_f1: 0.0125
state_f1: 0.0
theme_f1: 0.1755


In [ ]:
import json
import re
from collections import defaultdict

# =========================
# LOAD DATA
# =========================
GT_PATH = "/content/Testing_Ground_Truth_Data.txt"
PRED_PATH = "/content/flan_decomposed_output.json"

with open(GT_PATH, "r") as f:
    gt_data = json.load(f)

with open(PRED_PATH, "r") as f:
    pred_data = json.load(f)

print("GT:", len(gt_data), "Pred:", len(pred_data))

# =========================
# FIELDS
# =========================
FIELDS = ["emotion", "setting", "atmosphere", "state", "theme"]

# =========================
# NORMALIZATION
# =========================
def norm(x):
    if x is None:
        return ""
    return str(x).strip().lower()

# =========================
# STRICT JSON EXTRACTION
# =========================
def extract_fields(raw):
    """
    Robust extractor:
    - tries JSON parse
    - regex fallback
    - never silently drops fields
    """
    if raw is None:
        return {}

    if isinstance(raw, dict):
        return raw

    if isinstance(raw, list):
        return raw[0] if len(raw) > 0 else {}

    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except:
            pass

        out = {}
        for f in FIELDS:
            m = re.search(rf'"{f}"\s*:\s*"([^"]*)"', raw)
            if m:
                out[f] = m.group(1)
        return out

    return {}

# =========================
# METRICS STORAGE
# =========================
tp = defaultdict(int)
fp = defaultdict(int)
fn = defaultdict(int)

schema_valid = 0
total = len(gt_data)

hallucination_penalty = 0

# =========================
# MAIN LOOP
# =========================
for gt, pred in zip(gt_data, pred_data):

    gt_labels = {f: norm(gt.get(f)) for f in FIELDS}
    pred_raw = pred.get("raw_output", pred)
    pred_labels = extract_fields(pred_raw)
    pred_labels = {f: norm(pred_labels.get(f, "")) for f in FIELDS}

    # =========================
    # SCHEMA CHECK
    # =========================
    is_valid = all(pred_labels[f] != "" for f in FIELDS)

    if not is_valid:
        hallucination_penalty += 1
    else:
        schema_valid += 1

    # =========================
    # FIELD LEVEL SCORES
    # STRICT: missing = FN
    # =========================
    for f in FIELDS:
        gt_val = gt_labels[f]
        pred_val = pred_labels[f]

        if pred_val == "":
            fn[f] += 1
        elif pred_val == gt_val:
            tp[f] += 1
        else:
            fp[f] += 1
            fn[f] += 1

# =========================
# METRIC CALCULATIONS
# =========================
def precision(tp, fp):
    return tp / (tp + fp + 1e-9)

def recall(tp, fn):
    return tp / (tp + fn + 1e-9)

def f1(p, r):
    return 2 * p * r / (p + r + 1e-9)

# =========================
# FINAL RESULTS
# =========================
print("\n================ FINAL RESULTS (V2) ================\n")

results = {}

for f in FIELDS:
    p = precision(tp[f], fp[f])
    r = recall(tp[f], fn[f])
    f1_score = f1(p, r)

    results[f] = f1_score

    print(f"{f}_F1:", round(f1_score, 4))

# =========================
# SCHEMA SCORE
# =========================
schema_score = schema_valid / total

# =========================
# HALLUCINATION RATE
# =========================
hallucination_rate = hallucination_penalty / total

print("\nschema_accuracy:", round(schema_score, 4))
print("hallucination_rate:", round(hallucination_rate, 4))

# =========================
# SUMMARY TABLE (PAPER READY)
# =========================
print("\n================ SUMMARY TABLE ================\n")

print("| Field | F1 Score |")
print("|------|--------|")

for f in FIELDS:
    print(f"| {f} | {round(results[f], 4)} |")

print(f"| schema | {round(schema_score, 4)} |")
print(f"| hallucination | {round(hallucination_rate, 4)} |")

GT: 40 Pred: 40

================ FINAL RESULTS (V2) ================

emotion_F1: 0.0
setting_F1: 0.0
atmosphere_F1: 0.0
state_F1: 0.0
theme_F1: 0.0

schema_accuracy: 0.0
hallucination_rate: 1.0

================ SUMMARY TABLE ================

| Field | F1 Score |
|------|--------|
| emotion | 0.0 |
| setting | 0.0 |
| atmosphere | 0.0 |
| state | 0.0 |
| theme | 0.0 |
| schema | 0.0 |
| hallucination | 1.0 |


In [ ]:
import json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from collections import defaultdict

GT_PATH = "/content/Testing_Ground_Truth_Data.txt"
PRED_PATH = "/content/flan_decomposed_output.json"

# =========================
# LOAD DATA
# =========================
with open(GT_PATH, "r") as f:
    gt_data = json.load(f)

with open(PRED_PATH, "r") as f:
    pred_data = json.load(f)

print("GT:", len(gt_data), "Pred:", len(pred_data))

# =========================
# MODEL FOR SEMANTIC EMBEDDING
# =========================
model = SentenceTransformer("all-MiniLM-L6-v2")

FIELDS = ["emotion", "setting", "atmosphere", "state", "theme"]

# =========================
# BUILD GT FIELD SENTENCES
# =========================
def build_gt_text(sample, field):
    return f"{field}: {sample.get(field, '')}"

# =========================
# BUILD PRED TEXT
# =========================
def build_pred_text(pred):
    return str(pred.get("raw_output", ""))

# =========================
# STORAGE
# =========================
scores = defaultdict(list)

# =========================
# MAIN LOOP
# =========================
for gt, pred in zip(gt_data, pred_data):

    pred_text = build_pred_text(pred)
    pred_emb = model.encode(pred_text)

    for f in FIELDS:
        gt_text = build_gt_text(gt, f)
        gt_emb = model.encode(gt_text)

        sim = cosine_similarity([gt_emb], [pred_emb])[0][0]

        # clamp similarity to [0,1]
        sim = max(0.0, min(1.0, sim))

        scores[f].append(sim)

# =========================
# RESULTS
# =========================
print("\n================ FINAL NEURIPS SEMANTIC RESULTS ================\n")

for f in FIELDS:
    print(f"{f}_score:", round(np.mean(scores[f]), 4))

GT: 40 Pred: 40


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


================ FINAL NEURIPS SEMANTIC RESULTS ================

emotion_score: 0.3412
setting_score: 0.2843
atmosphere_score: 0.3561
state_score: 0.1143
theme_score: 0.3899


In [ ]:
import json
import re
import numpy as np
from collections import defaultdict

# =========================================================
# SAFE JSON EXTRACTION
# =========================================================

def extract_json(text):
    if text is None:
        return None

    text = str(text).replace("```json", "").replace("```", "")
    matches = re.findall(r"\{.*\}", text, re.DOTALL)

    for m in reversed(matches):
        try:
            return json.loads(m)
        except:
            continue
    return None


# =========================================================
# SCHEMA NORMALIZATION
# =========================================================

FIELDS = ["emotion", "setting", "atmosphere", "state", "theme"]

def normalize(gt):
    if isinstance(gt, dict) and "output" in gt:
        gt = gt["output"]

    if not isinstance(gt, list):
        return []

    return [
        {k: s.get(k, "") for k in FIELDS + ["state_change"]}
        for s in gt
    ]


# =========================================================
# MATCHING LOGIC (FOR P/R/F1)
# =========================================================

def match_counts(gt_list, pred_list):
    gt = gt_list.copy()
    pred = pred_list.copy()

    tp = 0
    for p in pred:
        if p in gt:
            tp += 1
            gt.remove(p)

    fp = len(pred) - tp
    fn = len(gt)

    return tp, fp, fn


def prf1(tp, fp, fn):
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return precision, recall, f1


# =========================================================
# ATTRIBUTE F1
# =========================================================

def attr_f1(gt, pred, key):
    gt_vals = [s.get(key, "") for s in gt]
    pred_vals = [s.get(key, "") for s in pred]

    tp, fp, fn = match_counts(gt_vals, pred_vals)
    return prf1(tp, fp, fn)[2]


# =========================================================
# METRICS
# =========================================================

def scene_acc(gt, pred):
    return min(len(gt), len(pred)) / (max(len(gt), len(pred)) + 1e-9)


def state_transition(gt, pred):
    gt_t = [s.get("state_change", "") for s in gt if s.get("state_change")]
    pr_t = [s.get("state_change", "") for s in pred if s.get("state_change")]

    tp, fp, fn = match_counts(gt_t, pr_t)
    return prf1(tp, fp, fn)[2]


def causal(gt, pred):
    gt_s = [s.get("state", "") for s in gt]
    pr_s = [s.get("state", "") for s in pred]

    n = min(len(gt_s), len(pr_s))
    if n == 0:
        return 0.0

    return sum(gt_s[i] == pr_s[i] for i in range(n)) / n


def schema(pred):
    req = ["emotion", "setting", "atmosphere", "state", "theme"]

    total = 0
    ok = 0

    for s in pred:
        for r in req:
            total += 1
            if s.get(r):
                ok += 1

    return ok / (total + 1e-9)


# =========================================================
# EVALUATION LOOP
# =========================================================

def evaluate(gt_data, pred_data):

    res = defaultdict(list)

    for gt, pred in zip(gt_data, pred_data):

        gt_s = normalize(gt)
        pred_json = extract_json(pred.get("output", pred))

        if pred_json is None:
            continue

        pred_s = normalize(pred_json)

        res["scene_acc"].append(scene_acc(gt_s, pred_s))

        for f in FIELDS:
            res[f].append(attr_f1(gt_s, pred_s, f))

        res["state_transition"].append(state_transition(gt_s, pred_s))
        res["causal"].append(causal(gt_s, pred_s))
        res["schema"].append(schema(pred_s))

    return {k: float(np.mean(v)) for k, v in res.items()}


# =========================================================
# RUN FLAN
# =========================================================

gt_path = "/content/Testing_Ground_Truth_Data.txt"
pred_path = "/content/flan_decomposed_output (2).json"

with open(gt_path) as f:
    gt_data = json.load(f)

with open(pred_path) as f:
    pred_data = json.load(f)

print("GT:", len(gt_data), "Pred:", len(pred_data))

metrics = evaluate(gt_data, pred_data)

print("\n===== FLAN RESULTS =====\n")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

GT: 40 Pred: 40

===== FLAN RESULTS =====

